In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Chargement des données
# On lit le fichier qu'on vient de créer
df = pd.read_csv('PL_10_ans.csv')

# On garde uniquement les colonnes utiles
df_ml = df[['HomeTeam', 'AwayTeam', 'FTR']].dropna() # dropna() enlève les lignes vides s'il y en a

# On fait l'encodage
encoder_equipes = LabelEncoder()
# On apprend TOUTES les équipes des 10 dernières années
toutes_les_equipes = pd.concat([df_ml['HomeTeam'], df_ml['AwayTeam']])
encoder_equipes.fit(toutes_les_equipes)

df_ml['HomeTeam_ID'] = encoder_equipes.transform(df_ml['HomeTeam'])
df_ml['AwayTeam_ID'] = encoder_equipes.transform(df_ml['AwayTeam'])

encoder_resultat = LabelEncoder()
df_ml['Resultat_ID'] = encoder_resultat.fit_transform(df_ml['FTR'])

#  On entraine le modéle
X = df_ml[['HomeTeam_ID', 'AwayTeam_ID']]
y = df_ml['Resultat_ID']

# On garde 20% pour le test (environ 760 matchs de test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele = RandomForestClassifier(n_estimators=100, random_state=42) # 100 arbres de décision
modele.fit(X_train, y_train)

#Affichage des résultat
precision = accuracy_score(y_test, modele.predict(X_test))
print(f"Nombre de matchs étudiés : {len(df_ml)}")
print(f"Précision de l'IA sur 10 ans : {precision * 100:.2f}%")

Nombre de matchs étudiés : 3800
Précision de l'IA sur 10 ans : 45.13%


In [ ]:
import numpy as np

def predire_match(equipe_domicile, equipe_exterieur):
    try:
        id_dom = encoder_equipes.transform([equipe_domicile])[0]
        id_ext = encoder_equipes.transform([equipe_exterieur])[0]
    except ValueError:
        print(f"Équipe inconnue, vérifiez l'orthographe : '{equipe_domicile}' ou '{equipe_exterieur}'")
        return

    prediction_id = modele.predict([[id_dom, id_ext]])[0]
    resultat = encoder_resultat.inverse_transform([prediction_id])[0]
    proba = modele.predict_proba([[id_dom, id_ext]])[0]

    idx = {label: i for i, label in enumerate(encoder_resultat.classes_)}

    if resultat == 'H':
        gagnant = equipe_domicile
        confiance = proba[idx['H']]
    elif resultat == 'A':
        gagnant = equipe_exterieur
        confiance = proba[idx['A']]
    else:
        gagnant = "Match Nul"
        confiance = proba[idx['D']]

    print(f"{equipe_domicile} vs {equipe_exterieur}")
    print(f"Résultat prédit : {gagnant} ({confiance*100:.1f}%)")
    print("-" * 20)

# Quelques matchs test
predire_match('Man City', 'Liverpool')
predire_match('Arsenal', 'Tottenham')
predire_match('Chelsea', 'Sheffield United')

 Man City vs Liverpool
 Vainqueur prédit : Man City
Confiance de l'IA : 55.9%
--------------------
 Arsenal vs Tottenham
 Vainqueur prédit : Match Nul
Confiance de l'IA : 76.0%
--------------------
 Chelsea vs Sheffield United
 Vainqueur prédit : Chelsea
Confiance de l'IA : 67.9%
--------------------


C:\Users\zied9\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\zied9\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\zied9\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\zied9\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-pac